In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from statsmodels.tsa.arima.model import ARIMA

In [ ]:
filename = '/content/drive/MyDrive/Hack-o-Week/Week 2/dataset.csv'
df = pd.read_csv(filename)

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.set_index('timestamp', inplace=True)
df.fillna(0, inplace=True)

In [ ]:
energy_cols = [
    'ceiling_fan_energy [kWh]',
    'lighting_energy [kWh]',
    'plug_load_energy [kWh]',
    'chilled_water_energy [kWh]',
    'fcu_fan_energy [kWh]'
]

In [ ]:
df['total_energy_kwh'] = df[energy_cols].sum(axis=1)

In [ ]:
df_hourly = pd.DataFrame()
df_hourly['total_energy_kwh'] = df['total_energy_kwh'].resample('h').sum()
df_hourly['wifi_connected_devices'] = df['wifi_connected_devices [number]'].resample('h').mean()

In [ ]:
df_hourly.fillna(0, inplace=True)

In [ ]:
# We split the data to test on the very last 24 hours
train_data = df_hourly.iloc[:-24]
test_data = df_hourly.iloc[-24:]
# Define ARIMA parameters (p,d,q) = (5,1,0)
# You can adjust these based on ACF/PACF analysis
model = ARIMA(train_data['total_energy_kwh'], order=(5,1,0))
model_fit = model.fit()
# Forecast the next 24 steps (hours)
forecast_result = model_fit.get_forecast(steps=24)
forecast_df = pd.DataFrame({
    'forecast': forecast_result.predicted_mean,
    'lower_bound': forecast_result.conf_int().iloc[:, 0],
    'upper_bound': forecast_result.conf_int().iloc[:, 1]
}, index=test_data.index)

In [ ]:
fig = go.Figure()

# Plot Historical Data (Last 5 Days for clarity)
fig.add_trace(go.Scatter(
    x=train_data.index[-120:],
    y=train_data['total_energy_kwh'].tail(120),
    mode='lines',
    name='History (Train)',
    line=dict(color='gray')
))

# Plot Actual Test Data
fig.add_trace(go.Scatter(
    x=test_data.index,
    y=test_data['total_energy_kwh'],
    mode='lines+markers',
    name='Actual Consumption',
    line=dict(color='blue')
))

# Plot Forecast
fig.add_trace(go.Scatter(
    x=forecast_df.index,
    y=forecast_df['forecast'],
    mode='lines',
    name='ARIMA Forecast',
    line=dict(color='red', dash='dash')
))

# Plot Confidence Intervals (Shaded Area)
fig.add_trace(go.Scatter(
    x=forecast_df.index,
    y=forecast_df['upper_bound'],
    mode='lines',
    line=dict(width=0),
    showlegend=False,
    hoverinfo='skip'
))

fig.add_trace(go.Scatter(
    x=forecast_df.index,
    y=forecast_df['lower_bound'],
    mode='lines',
    line=dict(width=0),
    fill='tonexty',
    fillcolor='rgba(255, 0, 0, 0.2)',
    name='95% Confidence Interval',
    hoverinfo='skip'
))

# Layout Styling
fig.update_layout(
    title='Room 1 Electricity Forecast Dashboard (Hourly)',
    xaxis_title='Time',
    yaxis_title='Total Energy (kWh)',
    template='plotly_white',
    hovermode="x unified"
)

fig.show()